In [6]:
# ============================================
# Load production feature-generation function
# ============================================

import sys
import os

# Add project root to Python path
PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from backend.feature_engineering import create_prediction_features

print("Production feature function loaded successfully.")

Production feature function loaded successfully.


In [3]:
import os
import pandas as pd

file_path = os.path.expanduser("~/Desktop/exascaleAi/Project/Project - 1/integrated_demand_weather_holidays.csv")

integrated_df = pd.read_csv(
    file_path,
    parse_dates=["Datetime", "Date"]
)

integrated_df = integrated_df.sort_values("Datetime").reset_index(drop=True)
integrated_df.head()

,Datetime,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,Total_Load,Dhanbad_Temperature,Dhanbad_Humidity,Dhanbad_CloudCover,Dhanbad_WindSpeed,Date,Holiday_Name,Holiday_Type,Scope,Is_Public_Holiday
0,2017-01-01 00:00:00,30999.493670,18170.212767,20013.493977,69183.200413,14.80,89.0,18.0,6.8,2017-01-01,No Holiday,NaN,NaN,0
1,2017-01-01 00:30:00,27396.455697,17883.282673,18490.602410,63770.340780,14.65,90.5,20.5,6.6,2017-01-01,No Holiday,NaN,NaN,0
2,2017-01-01 01:00:00,25407.594937,16627.355623,17476.626507,59511.577067,14.50,92.0,23.0,6.4,2017-01-01,No Holiday,NaN,NaN,0
3,2017-01-01 01:30:00,23906.835443,15529.483283,16609.156627,56045.475353,14.40,93.0,23.0,6.0,2017-01-01,No Holiday,NaN,NaN,0
4,2017-01-01 02:00:00,22474.936707,14767.173253,15870.843373,53112.953333,14.30,94.0,23.0,5.6,2017-01-01,No Holiday,NaN,NaN,0


In [4]:
import numpy as np

In [5]:
print("Rows:", len(integrated_df))
print("Start:", integrated_df["Datetime"].min())
print("End:", integrated_df["Datetime"].max())

Rows: 17472
Start: 2017-01-01 00:00:00
End: 2017-12-30 23:30:00


In [11]:
historical_load = integrated_df[
    [
        "Datetime",
        "F1_132KV_PowerConsumption",
        "F2_132KV_PowerConsumption",
        "F3_132KV_PowerConsumption",
        "Total_Load"
    ]
].copy()

historical_load.head()

,Datetime,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,Total_Load
0,2017-01-01 00:00:00,30999.493670,18170.212767,20013.493977,69183.200413
1,2017-01-01 00:30:00,27396.455697,17883.282673,18490.602410,63770.340780
2,2017-01-01 01:00:00,25407.594937,16627.355623,17476.626507,59511.577067
3,2017-01-01 01:30:00,23906.835443,15529.483283,16609.156627,56045.475353
4,2017-01-01 02:00:00,22474.936707,14767.173253,15870.843373,53112.953333


In [15]:
import os

os.makedirs("../Project - 1/data/production", exist_ok=True)

historical_load.to_csv(
    "../Project - 1/data/production/historical_load.csv",
    index=False
)

In [16]:
def create_prediction_features(
    prediction_datetime,
    historical_load,
    weather_data,
    holiday_data
):
    """
    Construct the 34 features required by the trained XGBoost model
    for a single prediction timestamp.
    """

    prediction_datetime = pd.Timestamp(prediction_datetime)

    # ---------------------------------------------------------
    # 1. Calendar features
    # ---------------------------------------------------------

    hour = prediction_datetime.hour
    minute = prediction_datetime.minute

    day_of_week = prediction_datetime.dayofweek
    day_of_month = prediction_datetime.day
    month = prediction_datetime.month
    day_of_year = prediction_datetime.dayofyear

    week_of_year = prediction_datetime.isocalendar().week

    is_weekend = int(day_of_week >= 5)

    time_slot = hour * 2 + minute // 30

    # ---------------------------------------------------------
    # 2. Cyclical features
    # ---------------------------------------------------------

    time_slot_sin = np.sin(
        2 * np.pi * time_slot / 48
    )

    time_slot_cos = np.cos(
        2 * np.pi * time_slot / 48
    )

    day_of_week_sin = np.sin(
        2 * np.pi * day_of_week / 7
    )

    day_of_week_cos = np.cos(
        2 * np.pi * day_of_week / 7
    )

    day_of_year_sin = np.sin(
        2 * np.pi * day_of_year / 365
    )

    day_of_year_cos = np.cos(
        2 * np.pi * day_of_year / 365
    )

    # ---------------------------------------------------------
    # 3. Historical demand
    # ---------------------------------------------------------

    load_series = (
        historical_load
        .set_index("Datetime")["Total_Load"]
        .sort_index()
    )

    lag_1_time = prediction_datetime - pd.Timedelta(minutes=30)
    lag_2_time = prediction_datetime - pd.Timedelta(hours=1)
    lag_4_time = prediction_datetime - pd.Timedelta(hours=2)

    lag_48_time = prediction_datetime - pd.Timedelta(hours=24)
    lag_96_time = prediction_datetime - pd.Timedelta(hours=48)
    lag_336_time = prediction_datetime - pd.Timedelta(days=7)

    lag_1 = load_series.get(lag_1_time, np.nan)
    lag_2 = load_series.get(lag_2_time, np.nan)
    lag_4 = load_series.get(lag_4_time, np.nan)

    lag_48 = load_series.get(lag_48_time, np.nan)
    lag_96 = load_series.get(lag_96_time, np.nan)
    lag_336 = load_series.get(lag_336_time, np.nan)

    # ---------------------------------------------------------
    # 4. Rolling historical features
    # ---------------------------------------------------------

    historical_window = load_series[
        load_series.index < prediction_datetime
    ]

    rolling_mean_1h = historical_window.tail(2).mean()
    rolling_mean_2h = historical_window.tail(4).mean()
    rolling_mean_6h = historical_window.tail(12).mean()
    rolling_mean_24h = historical_window.tail(48).mean()

    rolling_std_2h = historical_window.tail(4).std()
    rolling_std_24h = historical_window.tail(48).std()

    # ---------------------------------------------------------
    # 5. Historical change
    # ---------------------------------------------------------

    historical_change_30min = lag_1 - lag_2

    # ---------------------------------------------------------
    # 6. Holiday
    # ---------------------------------------------------------

    prediction_date = prediction_datetime.normalize()

    holiday_match = holiday_data[
        holiday_data["Date"] == prediction_date
    ]

    is_holiday = int(len(holiday_match) > 0)

    # ---------------------------------------------------------
    # 7. Weather
    # ---------------------------------------------------------

    weather_match = weather_data[
        weather_data["Datetime"] == prediction_datetime
    ]

    if len(weather_match) == 0:
        raise ValueError(
            f"No Dhanbad weather data found for {prediction_datetime}"
        )

    weather_row = weather_match.iloc[0]

    dhanbad_temperature = weather_row["Dhanbad_Temperature"]
    dhanbad_humidity = weather_row["Dhanbad_Humidity"]
    dhanbad_cloud_cover = weather_row["Dhanbad_CloudCover"]
    dhanbad_wind_speed = weather_row["Dhanbad_WindSpeed"]

    temperature_humidity = (
        dhanbad_temperature * dhanbad_humidity
    )

    # ---------------------------------------------------------
    # 8. Construct feature vector
    # ---------------------------------------------------------

    features = {
        "hour": hour,
        "minute": minute,
        "day_of_week": day_of_week,
        "day_of_month": day_of_month,
        "month": month,
        "day_of_year": day_of_year,
        "week_of_year": int(week_of_year),
        "is_weekend": is_weekend,
        "time_slot": time_slot,

        "time_slot_sin": time_slot_sin,
        "time_slot_cos": time_slot_cos,

        "day_of_week_sin": day_of_week_sin,
        "day_of_week_cos": day_of_week_cos,

        "day_of_year_sin": day_of_year_sin,
        "day_of_year_cos": day_of_year_cos,

        "is_holiday": is_holiday,

        "Dhanbad_Temperature": dhanbad_temperature,
        "Dhanbad_Humidity": dhanbad_humidity,
        "Dhanbad_CloudCover": dhanbad_cloud_cover,
        "Dhanbad_WindSpeed": dhanbad_wind_speed,

        "Temperature_Humidity": temperature_humidity,

        "lag_1": lag_1,
        "lag_2": lag_2,
        "lag_4": lag_4,
        "lag_48": lag_48,
        "lag_96": lag_96,
        "lag_336": lag_336,

        "rolling_mean_1h": rolling_mean_1h,
        "rolling_mean_2h": rolling_mean_2h,
        "rolling_mean_6h": rolling_mean_6h,
        "rolling_mean_24h": rolling_mean_24h,

        "rolling_std_2h": rolling_std_2h,
        "rolling_std_24h": rolling_std_24h,

        "historical_change_30min": historical_change_30min
    }

    return pd.DataFrame([features])

In [21]:
weather_30min = pd.read_csv("dhanbad_waether.csv")

print(weather_30min.columns.tolist())
print(weather_30min.head())

['latitude', 'longitude', 'elevation', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation']
           latitude            longitude                 elevation  \
0         23.796133             86.38477                     231.0   
1              time  temperature_2m (°C)  relative_humidity_2m (%)   
2  2017-01-01T00:00                 14.8                        89   
3  2017-01-01T01:00                 14.5                        92   
4  2017-01-01T02:00                 14.3                        94   

  utc_offset_seconds               timezone timezone_abbreviation  
0              19800           Asia/Kolkata              GMT+5:30  
1    cloud_cover (%)  wind_speed_10m (km/h)                   NaN  
2                 18                    6.8                   NaN  
3                 23                    6.4                   NaN  
4                 23                    5.6                   NaN  


In [27]:
weather_30min = pd.read_csv(
    "dhanbad_waether.csv",
    skiprows=1
)

weather_30min.head()

,23.796133,86.38477,231.0,19800,Asia/Kolkata,GMT+5:30
0,time,temperature_2m (°C),relative_humidity_2m (%),cloud_cover (%),wind_speed_10m (km/h),NaN
1,2017-01-01T00:00,14.8,89,18,6.8,NaN
2,2017-01-01T01:00,14.5,92,23,6.4,NaN
3,2017-01-01T02:00,14.3,94,23,5.6,NaN
4,2017-01-01T03:00,14.2,95,21,6.2,NaN


In [28]:
print(weather_30min.columns.tolist())

['23.796133', '86.38477', '231.0', '19800', 'Asia/Kolkata', 'GMT+5:30']


In [34]:
weather_30min = weather_30min.rename(columns={
    "time": "Datetime",
    "temperature_2m (°C)": "Dhanbad_Temperature",
    "relative_humidity_2m (%)": "Dhanbad_Humidity",
    "cloud_cover (%)": "Dhanbad_CloudCover",
    "wind_speed_10m (km/h)": "Dhanbad_WindSpeed"
})
weather_30min.head()

,23.796133,86.38477,231.0,19800,Asia/Kolkata,GMT+5:30
0,time,temperature_2m (°C),relative_humidity_2m (%),cloud_cover (%),wind_speed_10m (km/h),NaN
1,2017-01-01T00:00,14.8,89,18,6.8,NaN
2,2017-01-01T01:00,14.5,92,23,6.4,NaN
3,2017-01-01T02:00,14.3,94,23,5.6,NaN
4,2017-01-01T03:00,14.2,95,21,6.2,NaN


In [36]:
integrated_df = pd.read_csv(
    "integrated_demand_weather_holidays.csv",
    parse_dates=["Datetime", "Date"]
)

print(integrated_df.columns.tolist())

['Datetime', 'F1_132KV_PowerConsumption', 'F2_132KV_PowerConsumption', 'F3_132KV_PowerConsumption', 'Total_Load', 'Dhanbad_Temperature', 'Dhanbad_Humidity', 'Dhanbad_CloudCover', 'Dhanbad_WindSpeed', 'Date', 'Holiday_Name', 'Holiday_Type', 'Scope', 'Is_Public_Holiday']


In [37]:
weather_30min = integrated_df[
    [
        "Datetime",
        "Dhanbad_Temperature",
        "Dhanbad_Humidity",
        "Dhanbad_CloudCover",
        "Dhanbad_WindSpeed"
    ]
].copy()

In [38]:
print(weather_30min.shape)
print(weather_30min.head())
print(weather_30min.columns.tolist())

(17472, 5)
             Datetime  Dhanbad_Temperature  Dhanbad_Humidity  \
0 2017-01-01 00:00:00                14.80              89.0   
1 2017-01-01 00:30:00                14.65              90.5   
2 2017-01-01 01:00:00                14.50              92.0   
3 2017-01-01 01:30:00                14.40              93.0   
4 2017-01-01 02:00:00                14.30              94.0   

   Dhanbad_CloudCover  Dhanbad_WindSpeed  
0                18.0                6.8  
1                20.5                6.6  
2                23.0                6.4  
3                23.0                6.0  
4                23.0                5.6  
['Datetime', 'Dhanbad_Temperature', 'Dhanbad_Humidity', 'Dhanbad_CloudCover', 'Dhanbad_WindSpeed']


In [39]:
print(weather_30min.shape)
print(weather_30min.columns.tolist())

(17472, 5)
['Datetime', 'Dhanbad_Temperature', 'Dhanbad_Humidity', 'Dhanbad_CloudCover', 'Dhanbad_WindSpeed']


In [41]:
holiday_calendar = integrated_df[
    [
        "Date",
        "Holiday_Name",
        "Holiday_Type",
        "Scope",
        "Is_Public_Holiday"
    ]
].drop_duplicates(subset=["Date"])

In [42]:
import joblib
import pandas as pd
import numpy as np

MODEL_PATH = "final_xgboost_model.pkl"
METADATA_PATH = "feature_metadata.pkl"

final_xgb_model = joblib.load(MODEL_PATH)
feature_metadata = joblib.load(METADATA_PATH)

MODEL_FEATURES = feature_metadata["feature_columns"]

print("Model loaded successfully.")
print("Number of features:", len(MODEL_FEATURES))

Model loaded successfully.
Number of features: 34


In [43]:
integrated_df = pd.read_csv(
    "integrated_demand_weather_holidays.csv",
    parse_dates=["Datetime", "Date"]
)

integrated_df = (
    integrated_df
    .sort_values("Datetime")
    .reset_index(drop=True)
)

print("Dataset shape:", integrated_df.shape)
print("Start:", integrated_df["Datetime"].min())
print("End:", integrated_df["Datetime"].max())

Dataset shape: (17472, 14)
Start: 2017-01-01 00:00:00
End: 2017-12-30 23:30:00


In [44]:
required_columns = [
    "Datetime",
    "Total_Load",
    "Dhanbad_Temperature",
    "Dhanbad_Humidity",
    "Dhanbad_CloudCover",
    "Dhanbad_WindSpeed",
    "Is_Public_Holiday"
]

missing_columns = [
    col for col in required_columns
    if col not in integrated_df.columns
]

print("Missing columns:", missing_columns)

Missing columns: []


In [45]:
print("Model feature list:")
for i, feature in enumerate(MODEL_FEATURES, start=1):
    print(f"{i:02d}. {feature}")

Model feature list:
01. hour
02. minute
03. day_of_week
04. day_of_month
05. month
06. day_of_year
07. week_of_year
08. is_weekend
09. time_slot
10. time_slot_sin
11. time_slot_cos
12. day_of_week_sin
13. day_of_week_cos
14. day_of_year_sin
15. day_of_year_cos
16. is_holiday
17. Dhanbad_Temperature
18. Dhanbad_Humidity
19. Dhanbad_CloudCover
20. Dhanbad_WindSpeed
21. Temperature_Humidity
22. lag_1
23. lag_2
24. lag_4
25. lag_48
26. lag_96
27. lag_336
28. rolling_mean_1h
29. rolling_mean_2h
30. rolling_mean_6h
31. rolling_mean_24h
32. rolling_std_2h
33. rolling_std_24h
34. historical_change_30min


In [46]:
print("Total model features:", len(MODEL_FEATURES))

Total model features: 34


In [47]:
print("Number of features:", len(MODEL_FEATURES))
print("Missing columns:", missing_columns)

Number of features: 34
Missing columns: []


In [48]:
def create_prediction_features(
    prediction_datetime,
    integrated_df
):
    """
    Create the exact 34 features required by the trained XGBoost model
    for a single prediction timestamp.

    The function uses:
    - historical Total_Load from integrated_df
    - Dhanbad weather from integrated_df
    - holiday information from integrated_df
    - calendar information calculated from prediction_datetime
    """

    prediction_datetime = pd.Timestamp(prediction_datetime)

    # ---------------------------------------------------------
    # 1. Historical load series
    # ---------------------------------------------------------

    load_series = (
        integrated_df
        .set_index("Datetime")["Total_Load"]
        .sort_index()
    )

    # ---------------------------------------------------------
    # 2. Calendar features
    # ---------------------------------------------------------

    hour = prediction_datetime.hour
    minute = prediction_datetime.minute

    day_of_week = prediction_datetime.dayofweek
    day_of_month = prediction_datetime.day
    month = prediction_datetime.month
    day_of_year = prediction_datetime.dayofyear

    week_of_year = int(prediction_datetime.isocalendar().week)

    is_weekend = int(day_of_week >= 5)

    time_slot = hour * 2 + minute // 30

    # ---------------------------------------------------------
    # 3. Cyclical features
    # ---------------------------------------------------------

    time_slot_sin = np.sin(
        2 * np.pi * time_slot / 48
    )

    time_slot_cos = np.cos(
        2 * np.pi * time_slot / 48
    )

    day_of_week_sin = np.sin(
        2 * np.pi * day_of_week / 7
    )

    day_of_week_cos = np.cos(
        2 * np.pi * day_of_week / 7
    )

    day_of_year_sin = np.sin(
        2 * np.pi * day_of_year / 365
    )

    day_of_year_cos = np.cos(
        2 * np.pi * day_of_year / 365
    )

    # ---------------------------------------------------------
    # 4. Historical demand lags
    # ---------------------------------------------------------

    lag_1 = load_series.get(
        prediction_datetime - pd.Timedelta(minutes=30),
        np.nan
    )

    lag_2 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=1),
        np.nan
    )

    lag_4 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=2),
        np.nan
    )

    lag_48 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=24),
        np.nan
    )

    lag_96 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=48),
        np.nan
    )

    lag_336 = load_series.get(
        prediction_datetime - pd.Timedelta(days=7),
        np.nan
    )

    # ---------------------------------------------------------
    # 5. Historical rolling features
    #
    # IMPORTANT:
    # Only observations BEFORE the prediction timestamp
    # are used.
    # ---------------------------------------------------------

    historical_window = load_series[
        load_series.index < prediction_datetime
    ]

    rolling_mean_1h = historical_window.tail(2).mean()
    rolling_mean_2h = historical_window.tail(4).mean()
    rolling_mean_6h = historical_window.tail(12).mean()
    rolling_mean_24h = historical_window.tail(48).mean()

    rolling_std_2h = historical_window.tail(4).std()
    rolling_std_24h = historical_window.tail(48).std()

    # ---------------------------------------------------------
    # 6. Historical change
    # ---------------------------------------------------------

    historical_change_30min = lag_1 - lag_2

    # ---------------------------------------------------------
    # 7. Holiday information
    # ---------------------------------------------------------

    prediction_date = prediction_datetime.normalize()

    holiday_rows = integrated_df[
        integrated_df["Date"] == prediction_date
    ]

    if len(holiday_rows) > 0:
        is_holiday = int(
            holiday_rows["Is_Public_Holiday"].max()
        )
    else:
        is_holiday = 0

    # ---------------------------------------------------------
    # 8. Dhanbad weather
    # ---------------------------------------------------------

    weather_rows = integrated_df[
        integrated_df["Datetime"] == prediction_datetime
    ]

    if len(weather_rows) == 0:
        raise ValueError(
            f"No Dhanbad weather data found for "
            f"{prediction_datetime}"
        )

    weather_row = weather_rows.iloc[0]

    dhanbad_temperature = weather_row[
        "Dhanbad_Temperature"
    ]

    dhanbad_humidity = weather_row[
        "Dhanbad_Humidity"
    ]

    dhanbad_cloud_cover = weather_row[
        "Dhanbad_CloudCover"
    ]

    dhanbad_wind_speed = weather_row[
        "Dhanbad_WindSpeed"
    ]

    # ---------------------------------------------------------
    # 9. Weather interaction
    # ---------------------------------------------------------

    temperature_humidity = (
        dhanbad_temperature *
        dhanbad_humidity
    )

    # ---------------------------------------------------------
    # 10. Construct the feature dictionary
    # ---------------------------------------------------------

    features = {
        "hour": hour,
        "minute": minute,
        "day_of_week": day_of_week,
        "day_of_month": day_of_month,
        "month": month,
        "day_of_year": day_of_year,
        "week_of_year": week_of_year,
        "is_weekend": is_weekend,
        "time_slot": time_slot,

        "time_slot_sin": time_slot_sin,
        "time_slot_cos": time_slot_cos,

        "day_of_week_sin": day_of_week_sin,
        "day_of_week_cos": day_of_week_cos,

        "day_of_year_sin": day_of_year_sin,
        "day_of_year_cos": day_of_year_cos,

        "is_holiday": is_holiday,

        "Dhanbad_Temperature": dhanbad_temperature,
        "Dhanbad_Humidity": dhanbad_humidity,
        "Dhanbad_CloudCover": dhanbad_cloud_cover,
        "Dhanbad_WindSpeed": dhanbad_wind_speed,

        "Temperature_Humidity": temperature_humidity,

        "lag_1": lag_1,
        "lag_2": lag_2,
        "lag_4": lag_4,
        "lag_48": lag_48,
        "lag_96": lag_96,
        "lag_336": lag_336,

        "rolling_mean_1h": rolling_mean_1h,
        "rolling_mean_2h": rolling_mean_2h,
        "rolling_mean_6h": rolling_mean_6h,
        "rolling_mean_24h": rolling_mean_24h,

        "rolling_std_2h": rolling_std_2h,
        "rolling_std_24h": rolling_std_24h,

        "historical_change_30min": historical_change_30min
    }

    # ---------------------------------------------------------
    # 11. Convert to DataFrame
    # ---------------------------------------------------------

    feature_df = pd.DataFrame([features])

    # ---------------------------------------------------------
    # 12. Force exact training feature order
    # ---------------------------------------------------------

    feature_df = feature_df[MODEL_FEATURES]

    return feature_df

In [49]:
prediction_time = pd.Timestamp(
    "2017-12-15 18:00:00"
)

In [50]:
prediction_features = create_prediction_features(
    prediction_datetime=prediction_time,
    integrated_df=integrated_df
)

prediction_features

,hour,minute,day_of_week,day_of_month,month,day_of_year,week_of_year,is_weekend,time_slot,time_slot_sin,...,lag_48,lag_96,lag_336,rolling_mean_1h,rolling_mean_2h,rolling_mean_6h,rolling_mean_24h,rolling_std_2h,rolling_std_24h,historical_change_30min
0,18,0,4,15,12,349,50,0,36,-1.0,...,90118.70168,89474.423813,88315.881513,81539.72388,73797.822736,68537.55468,64735.280891,9939.252295,14203.396755,10613.581127


In [51]:
print("Shape:", prediction_features.shape)

Shape: (1, 34)


In [52]:
print(
    prediction_features.columns.tolist()
    == MODEL_FEATURES
)

True


In [53]:
prediction_features.isna().sum()

hour                       0
minute                     0
day_of_week                0
day_of_month               0
month                      0
day_of_year                0
week_of_year               0
is_weekend                 0
time_slot                  0
time_slot_sin              0
time_slot_cos              0
day_of_week_sin            0
day_of_week_cos            0
day_of_year_sin            0
day_of_year_cos            0
is_holiday                 0
Dhanbad_Temperature        0
Dhanbad_Humidity           0
Dhanbad_CloudCover         0
Dhanbad_WindSpeed          0
Temperature_Humidity       0
lag_1                      0
lag_2                      0
lag_4                      0
lag_48                     0
lag_96                     0
lag_336                    0
rolling_mean_1h            0
rolling_mean_2h            0
rolling_mean_6h            0
rolling_mean_24h           0
rolling_std_2h             0
rolling_std_24h            0
historical_change_30min    0
dtype: int64

In [54]:
print(
    "Missing values:",
    prediction_features.isna().sum().sum()
)

Missing values: 0


In [55]:
production_prediction = final_xgb_model.predict(
    prediction_features
)[0]

print(
    f"Predicted Total Load: {production_prediction:,.2f}"
)

Predicted Total Load: 89,780.12


In [56]:
actual_load = integrated_df.loc[
    integrated_df["Datetime"] == prediction_time,
    "Total_Load"
].iloc[0]

absolute_error = abs(
    actual_load - production_prediction
)

percentage_error = (
    absolute_error / actual_load
) * 100

print(f"Prediction time : {prediction_time}")
print(f"Actual load     : {actual_load:,.2f}")
print(f"Predicted load  : {production_prediction:,.2f}")
print(f"Absolute error  : {absolute_error:,.2f}")
print(f"Percentage error: {percentage_error:.2f}%")

Prediction time : 2017-12-15 18:00:00
Actual load     : 86,813.43
Predicted load  : 89,780.12
Absolute error  : 2,966.70
Percentage error: 3.42%


In [57]:
print("Shape:", prediction_features.shape)
print("Feature order correct:", prediction_features.columns.tolist() == MODEL_FEATURES)
print("Missing values:", prediction_features.isna().sum().sum())

print(f"Actual: {actual_load:,.2f}")
print(f"Predicted: {production_prediction:,.2f}")
print(f"Error: {absolute_error:,.2f}")

Shape: (1, 34)
Feature order correct: True
Missing values: 0
Actual: 86,813.43
Predicted: 89,780.12
Error: 2,966.70


In [58]:
test_timestamps = [
    "2017-12-15 06:00:00",
    "2017-12-15 12:00:00",
    "2017-12-15 18:00:00",
    "2017-12-16 20:00:00",
    "2017-12-25 18:00:00"
]

production_validation = []

for timestamp in test_timestamps:

    timestamp = pd.Timestamp(timestamp)

    features = create_prediction_features(
        prediction_datetime=timestamp,
        integrated_df=integrated_df
    )

    prediction = final_xgb_model.predict(features)[0]

    actual_rows = integrated_df[
        integrated_df["Datetime"] == timestamp
    ]

    if len(actual_rows) > 0:

        actual = actual_rows["Total_Load"].iloc[0]

        absolute_error = abs(
            actual - prediction
        )

        percentage_error = (
            absolute_error / actual
        ) * 100

    else:
        actual = np.nan
        absolute_error = np.nan
        percentage_error = np.nan

    production_validation.append({
        "Datetime": timestamp,
        "Actual_Load": actual,
        "Predicted_Load": prediction,
        "Absolute_Error": absolute_error,
        "Percentage_Error": percentage_error
    })

production_validation_df = pd.DataFrame(
    production_validation
)

production_validation_df

,Datetime,Actual_Load,Predicted_Load,Absolute_Error,Percentage_Error
0,2017-12-15 06:00:00,51269.718234,51330.355469,60.637234,0.118271
1,2017-12-15 12:00:00,66292.002567,67913.062500,1621.059933,2.445333
2,2017-12-15 18:00:00,86813.429830,89780.125000,2966.695170,3.417323
3,2017-12-16 20:00:00,84902.925690,85584.539062,681.613372,0.802815
4,2017-12-25 18:00:00,91673.910833,92924.367188,1250.456354,1.364026


In [59]:
holiday_test_features = create_prediction_features(
    prediction_datetime=pd.Timestamp("2017-12-25 18:00:00"),
    integrated_df=integrated_df
)

print(
    "is_holiday:",
    holiday_test_features["is_holiday"].iloc[0]
)

is_holiday: 1


In [60]:
normal_test_features = create_prediction_features(
    prediction_datetime=pd.Timestamp("2017-12-15 18:00:00"),
    integrated_df=integrated_df
)

print(
    "is_holiday:",
    normal_test_features["is_holiday"].iloc[0]
)

is_holiday: 0


In [61]:
weather_check = create_prediction_features(
    prediction_datetime=pd.Timestamp("2017-12-15 18:00:00"),
    integrated_df=integrated_df
)

weather_check[
    [
        "Dhanbad_Temperature",
        "Dhanbad_Humidity",
        "Dhanbad_CloudCover",
        "Dhanbad_WindSpeed",
        "Temperature_Humidity"
    ]
]

,Dhanbad_Temperature,Dhanbad_Humidity,Dhanbad_CloudCover,Dhanbad_WindSpeed,Temperature_Humidity
0,21.1,65.0,88.0,7.9,1371.5


## Production Feature Pipeline Validation

The production feature-generation pipeline was tested independently from the
model development process.

For each requested timestamp, the pipeline automatically constructs the
complete set of 34 features required by the trained XGBoost model.

The validation confirmed that:

- The generated feature matrix contains exactly 34 features.
- Feature names and ordering exactly match the training feature metadata.
- No feature values are missing for valid historical timestamps.
- Historical demand features are generated from previously observed load.
- Calendar features are generated automatically from the requested timestamp.
- Dhanbad-specific weather variables are retrieved from the integrated weather
  dataset.
- Public-holiday status is determined automatically from the holiday calendar.
- The resulting feature vector can be passed directly to the saved XGBoost
  model.

A single historical prediction is used only as a pipeline sanity check and is
not treated as a replacement for the formal test-set evaluation reported in
the model development notebook.

In [62]:
def create_prediction_features(
    prediction_datetime,
    integrated_df
):
    """
    Create the 34 features required by the trained XGBoost model.

    Raises a clear ValueError if:
    - timestamp is not on a 30-minute boundary
    - required historical data is unavailable
    - weather data is unavailable
    - generated features contain missing values
    """

    prediction_datetime = pd.Timestamp(prediction_datetime)

    # ---------------------------------------------------------
    # 1. Validate timestamp
    # ---------------------------------------------------------

    if prediction_datetime.minute not in [0, 30]:
        raise ValueError(
            "Prediction time must be aligned to a 30-minute interval."
        )

    if prediction_datetime.second != 0:
        raise ValueError(
            "Prediction timestamp must have zero seconds."
        )

    # ---------------------------------------------------------
    # 2. Prepare historical load series
    # ---------------------------------------------------------

    load_series = (
        integrated_df
        .set_index("Datetime")["Total_Load"]
        .sort_index()
    )

    # ---------------------------------------------------------
    # 3. Check required historical range
    # ---------------------------------------------------------

    earliest_required_time = (
        prediction_datetime - pd.Timedelta(days=7)
    )

    available_history = load_series[
        load_series.index <= earliest_required_time
    ]

    if len(available_history) == 0:
        raise ValueError(
            f"Insufficient historical data for {prediction_datetime}. "
            "The model requires at least 7 days of prior demand history."
        )

    # Check specifically that the required 7-day lag exists
    lag_336_time = (
        prediction_datetime - pd.Timedelta(days=7)
    )

    if lag_336_time not in load_series.index:
        raise ValueError(
            f"Required 7-day historical demand is unavailable for "
            f"{prediction_datetime}."
        )

    # ---------------------------------------------------------
    # 4. Calendar features
    # ---------------------------------------------------------

    hour = prediction_datetime.hour
    minute = prediction_datetime.minute

    day_of_week = prediction_datetime.dayofweek
    day_of_month = prediction_datetime.day
    month = prediction_datetime.month
    day_of_year = prediction_datetime.dayofyear

    week_of_year = int(
        prediction_datetime.isocalendar().week
    )

    is_weekend = int(day_of_week >= 5)

    time_slot = (
        hour * 2 + minute // 30
    )

    # ---------------------------------------------------------
    # 5. Cyclical features
    # ---------------------------------------------------------

    time_slot_sin = np.sin(
        2 * np.pi * time_slot / 48
    )

    time_slot_cos = np.cos(
        2 * np.pi * time_slot / 48
    )

    day_of_week_sin = np.sin(
        2 * np.pi * day_of_week / 7
    )

    day_of_week_cos = np.cos(
        2 * np.pi * day_of_week / 7
    )

    day_of_year_sin = np.sin(
        2 * np.pi * day_of_year / 365
    )

    day_of_year_cos = np.cos(
        2 * np.pi * day_of_year / 365
    )

    # ---------------------------------------------------------
    # 6. Historical lags
    # ---------------------------------------------------------

    lag_1 = load_series.get(
        prediction_datetime - pd.Timedelta(minutes=30),
        np.nan
    )

    lag_2 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=1),
        np.nan
    )

    lag_4 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=2),
        np.nan
    )

    lag_48 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=24),
        np.nan
    )

    lag_96 = load_series.get(
        prediction_datetime - pd.Timedelta(hours=48),
        np.nan
    )

    lag_336 = load_series.get(
        prediction_datetime - pd.Timedelta(days=7),
        np.nan
    )

    # ---------------------------------------------------------
    # 7. Verify lag availability
    # ---------------------------------------------------------

    lag_values = {
        "lag_1": lag_1,
        "lag_2": lag_2,
        "lag_4": lag_4,
        "lag_48": lag_48,
        "lag_96": lag_96,
        "lag_336": lag_336
    }

    missing_lags = [
        name
        for name, value in lag_values.items()
        if pd.isna(value)
    ]

    if missing_lags:
        raise ValueError(
            "Missing historical demand features: "
            + ", ".join(missing_lags)
        )

    # ---------------------------------------------------------
    # 8. Rolling features
    # ---------------------------------------------------------

    historical_window = load_series[
        load_series.index < prediction_datetime
    ]

    if len(historical_window) < 48:
        raise ValueError(
            "Insufficient historical observations to calculate "
            "24-hour rolling features."
        )

    rolling_mean_1h = (
        historical_window.tail(2).mean()
    )

    rolling_mean_2h = (
        historical_window.tail(4).mean()
    )

    rolling_mean_6h = (
        historical_window.tail(12).mean()
    )

    rolling_mean_24h = (
        historical_window.tail(48).mean()
    )

    rolling_std_2h = (
        historical_window.tail(4).std()
    )

    rolling_std_24h = (
        historical_window.tail(48).std()
    )

    # ---------------------------------------------------------
    # 9. Historical change
    # ---------------------------------------------------------

    historical_change_30min = (
        lag_1 - lag_2
    )

    # ---------------------------------------------------------
    # 10. Holiday
    # ---------------------------------------------------------

    prediction_date = (
        prediction_datetime.normalize()
    )

    holiday_rows = integrated_df[
        integrated_df["Date"] == prediction_date
    ]

    if len(holiday_rows) > 0:
        is_holiday = int(
            holiday_rows["Is_Public_Holiday"].max()
        )
    else:
        is_holiday = 0

    # ---------------------------------------------------------
    # 11. Dhanbad weather
    # ---------------------------------------------------------

    weather_rows = integrated_df[
        integrated_df["Datetime"] == prediction_datetime
    ]

    if len(weather_rows) == 0:
        raise ValueError(
            f"No Dhanbad weather data is available for "
            f"{prediction_datetime}."
        )

    weather_row = weather_rows.iloc[0]

    dhanbad_temperature = weather_row[
        "Dhanbad_Temperature"
    ]

    dhanbad_humidity = weather_row[
        "Dhanbad_Humidity"
    ]

    dhanbad_cloud_cover = weather_row[
        "Dhanbad_CloudCover"
    ]

    dhanbad_wind_speed = weather_row[
        "Dhanbad_WindSpeed"
    ]

    # ---------------------------------------------------------
    # 12. Weather interaction
    # ---------------------------------------------------------

    temperature_humidity = (
        dhanbad_temperature *
        dhanbad_humidity
    )

    # ---------------------------------------------------------
    # 13. Construct features
    # ---------------------------------------------------------

    features = {
        "hour": hour,
        "minute": minute,
        "day_of_week": day_of_week,
        "day_of_month": day_of_month,
        "month": month,
        "day_of_year": day_of_year,
        "week_of_year": week_of_year,
        "is_weekend": is_weekend,
        "time_slot": time_slot,

        "time_slot_sin": time_slot_sin,
        "time_slot_cos": time_slot_cos,

        "day_of_week_sin": day_of_week_sin,
        "day_of_week_cos": day_of_week_cos,

        "day_of_year_sin": day_of_year_sin,
        "day_of_year_cos": day_of_year_cos,

        "is_holiday": is_holiday,

        "Dhanbad_Temperature": dhanbad_temperature,
        "Dhanbad_Humidity": dhanbad_humidity,
        "Dhanbad_CloudCover": dhanbad_cloud_cover,
        "Dhanbad_WindSpeed": dhanbad_wind_speed,

        "Temperature_Humidity": temperature_humidity,

        "lag_1": lag_1,
        "lag_2": lag_2,
        "lag_4": lag_4,
        "lag_48": lag_48,
        "lag_96": lag_96,
        "lag_336": lag_336,

        "rolling_mean_1h": rolling_mean_1h,
        "rolling_mean_2h": rolling_mean_2h,
        "rolling_mean_6h": rolling_mean_6h,
        "rolling_mean_24h": rolling_mean_24h,

        "rolling_std_2h": rolling_std_2h,
        "rolling_std_24h": rolling_std_24h,

        "historical_change_30min": historical_change_30min
    }

    feature_df = pd.DataFrame([features])

    # ---------------------------------------------------------
    # 14. Enforce exact training feature order
    # ---------------------------------------------------------

    feature_df = feature_df[
        MODEL_FEATURES
    ]

    # ---------------------------------------------------------
    # 15. Final validation
    # ---------------------------------------------------------

    missing_features = feature_df.columns[
        feature_df.isna().any()
    ].tolist()

    if missing_features:
        raise ValueError(
            "Generated features contain missing values: "
            + ", ".join(missing_features)
        )

    if feature_df.shape[1] != 34:
        raise ValueError(
            f"Expected 34 features but generated "
            f"{feature_df.shape[1]}."
        )

    return feature_df

In [63]:
prediction_time = pd.Timestamp(
    "2017-12-15 18:00:00"
)

prediction_features = create_prediction_features(
    prediction_time,
    integrated_df
)

print("Shape:", prediction_features.shape)
print(
    "Missing values:",
    prediction_features.isna().sum().sum()
)

Shape: (1, 34)
Missing values: 0


In [64]:
prediction = final_xgb_model.predict(
    prediction_features
)[0]

print(f"Prediction: {prediction:,.2f}")

Prediction: 89,780.12


In [65]:
try:

    create_prediction_features(
        "2017-12-15 18:15:00",
        integrated_df
    )

except ValueError as e:

    print("Validation error:", e)

Validation error: Prediction time must be aligned to a 30-minute interval.


In [66]:
try:

    create_prediction_features(
        "2017-01-03 18:00:00",
        integrated_df
    )

except ValueError as e:

    print("Validation error:", e)

Validation error: Insufficient historical data for 2017-01-03 18:00:00. The model requires at least 7 days of prior demand history.


In [67]:
test_data = integrated_df.copy()

test_data = test_data[
    test_data["Datetime"] != pd.Timestamp(
        "2017-12-15 18:00:00"
    )
]

In [68]:
try:

    create_prediction_features(
        "2017-12-15 18:00:00",
        test_data
    )

except ValueError as e:

    print("Validation error:", e)

Validation error: No Dhanbad weather data is available for 2017-12-15 18:00:00.


## Production Input Validation

The feature-generation pipeline includes validation before a prediction is
passed to the trained XGBoost model.

The validation ensures that:

- Prediction timestamps are aligned to the 30-minute forecasting interval.
- At least seven days of historical demand are available for the required
  `lag_336` feature.
- All required historical lag features are available.
- Sufficient observations exist for the rolling features.
- Dhanbad weather data is available for the requested timestamp.
- All 34 generated features contain valid values.
- The feature order exactly matches the order used during model training.

Invalid requests are rejected with descriptive errors instead of allowing
incomplete feature vectors to reach the prediction model.

This validation logic will later be incorporated into the FastAPI backend.

In [3]:
# ============================================
# Production vs Training Feature Parity Check
# ============================================

import os
import joblib
import numpy as np
import pandas as pd

# --------------------------------------------
# 1. Load saved model metadata
# --------------------------------------------

METADATA_PATH = "models/feature_metadata.pkl"

metadata = joblib.load(METADATA_PATH)

feature_columns = metadata["feature_columns"]

print("Number of model features:", len(feature_columns))
print("\nFeature columns:")
for i, feature in enumerate(feature_columns, start=1):
    print(f"{i:2d}. {feature}")

Number of model features: 34

Feature columns:
 1. hour
 2. minute
 3. day_of_week
 4. day_of_month
 5. month
 6. day_of_year
 7. week_of_year
 8. is_weekend
 9. time_slot
10. time_slot_sin
11. time_slot_cos
12. day_of_week_sin
13. day_of_week_cos
14. day_of_year_sin
15. day_of_year_cos
16. is_holiday
17. Dhanbad_Temperature
18. Dhanbad_Humidity
19. Dhanbad_CloudCover
20. Dhanbad_WindSpeed
21. Temperature_Humidity
22. lag_1
23. lag_2
24. lag_4
25. lag_48
26. lag_96
27. lag_336
28. rolling_mean_1h
29. rolling_mean_2h
30. rolling_mean_6h
31. rolling_mean_24h
32. rolling_std_2h
33. rolling_std_24h
34. historical_change_30min


In [4]:
# ============================================
# 2. Load saved train / validation / test data
# ============================================

train_features = pd.read_csv(
    "data/processed/train_features.csv",
    parse_dates=["Datetime", "Date"]
)

validation_features = pd.read_csv(
    "data/processed/validation_features.csv",
    parse_dates=["Datetime", "Date"]
)

test_features = pd.read_csv(
    "data/processed/test_features.csv",
    parse_dates=["Datetime", "Date"]
)

saved_features = pd.concat(
    [
        train_features,
        validation_features,
        test_features
    ],
    ignore_index=True
)

print("Train shape:", train_features.shape)
print("Validation shape:", validation_features.shape)
print("Test shape:", test_features.shape)
print("Combined shape:", saved_features.shape)

Train shape: (11952, 44)
Validation shape: (2544, 44)
Test shape: (2640, 44)
Combined shape: (17136, 44)


In [11]:
# ============================================
# 3. Production vs Training Feature Parity
# ============================================

test_timestamps = [
    "2017-01-08 00:00:00",
    "2017-03-15 12:30:00",
    "2017-06-20 18:00:00",
    "2017-09-15 09:30:00",
    "2017-11-10 20:00:00",
    "2017-12-15 18:00:00"
]

parity_results = []

for timestamp in test_timestamps:

    timestamp = pd.Timestamp(timestamp)

    # ----------------------------------------
    # Saved training feature row
    # ----------------------------------------

    saved_row = saved_features[
        saved_features["Datetime"] == timestamp
    ]

    if saved_row.empty:
        print(f"Timestamp not found: {timestamp}")
        continue

    # ----------------------------------------
    # Generate features using production logic
    # ----------------------------------------

    production_row = create_prediction_features(
        timestamp,
        integrated_df,
        feature_columns
    )

    # ----------------------------------------
    # Extract feature values
    # ----------------------------------------

    saved_values = (
        saved_row.iloc[0][feature_columns]
        .astype(float)
    )

    production_values = (
        production_row.iloc[0][feature_columns]
        .astype(float)
    )

    # ----------------------------------------
    # Calculate differences
    # ----------------------------------------

    absolute_difference = np.abs(
        saved_values.values - production_values.values
    )

    max_difference = absolute_difference.max()
    mean_difference = absolute_difference.mean()

    mismatched_features = [
        feature_columns[i]
        for i, diff in enumerate(absolute_difference)
        if diff > 1e-6
    ]

    parity_results.append({
        "Datetime": timestamp,
        "Max Absolute Difference": max_difference,
        "Mean Absolute Difference": mean_difference,
        "Mismatched Features": len(mismatched_features),
        "Status": (
            "PASS"
            if max_difference <= 1e-6
            else "FAIL"
        )
    })

parity_df = pd.DataFrame(parity_results)

parity_df

,Datetime,Max Absolute Difference,Mean Absolute Difference,Mismatched Features,Status
0,2017-01-08 00:00:00,3.674359e-10,1.091394e-11,0,PASS
1,2017-03-15 12:30:00,1.383796e-09,4.343506e-11,0,PASS
2,2017-06-20 18:00:00,6.057917e-09,2.025766e-10,0,PASS
3,2017-09-15 09:30:00,4.465619e-09,1.560051e-10,0,PASS
4,2017-11-10 20:00:00,2.867819e-08,8.574395e-10,0,PASS
5,2017-12-15 18:00:00,6.348273e-10,3.022733e-11,0,PASS


In [7]:
# ============================================
# Load integrated production dataset
# ============================================

integrated_df = pd.read_csv(
    "data/processed/integrated_demand_weather_holidays.csv",
    parse_dates=["Datetime", "Date"]
)

integrated_df = (
    integrated_df
    .sort_values("Datetime")
    .reset_index(drop=True)
)

print("Integrated data shape:", integrated_df.shape)
print("Date range:")
print(integrated_df["Datetime"].min())
print("to")
print(integrated_df["Datetime"].max())

Integrated data shape: (17472, 14)
Date range:
2017-01-01 00:00:00
to
2017-12-30 23:30:00


In [9]:
# ============================================
# 4. Final Parity Assertion
# ============================================

if (parity_df["Status"] == "PASS").all():

    print("============================================")
    print("PRODUCTION FEATURE PARITY CHECK PASSED")
    print("============================================")
    print("Production features match saved training features.")

else:

    print("============================================")
    print("PRODUCTION FEATURE PARITY CHECK FAILED")
    print("============================================")

    print("\nFailed timestamps:")
    print(
        parity_df[
            parity_df["Status"] == "FAIL"
        ]
    )

PRODUCTION FEATURE PARITY CHECK FAILED

Failed timestamps:
             Datetime  Max Absolute Difference  Mean Absolute Difference  \
4 2017-11-10 20:00:00             2.867819e-08              8.574395e-10   

   Mismatched Features Status  
4                    1   FAIL  


In [10]:
# ============================================
# Identify the exact feature causing the mismatch
# ============================================

timestamp = pd.Timestamp("2017-11-10 20:00:00")

saved_row = saved_features[
    saved_features["Datetime"] == timestamp
].iloc[0]

production_row = create_prediction_features(
    timestamp,
    integrated_df,
    feature_columns
).iloc[0]

comparison = pd.DataFrame({
    "feature": feature_columns,
    "saved_value": saved_row[feature_columns].astype(float).values,
    "production_value": production_row[feature_columns].astype(float).values
})

comparison["absolute_difference"] = (
    comparison["saved_value"] -
    comparison["production_value"]
).abs()

comparison[
    comparison["absolute_difference"] > 1e-10
].sort_values(
    "absolute_difference",
    ascending=False
)

,feature,saved_value,production_value,absolute_difference
31,rolling_std_2h,1027.573604,1027.573603,2.867819e-08
32,rolling_std_24h,13887.495885,13887.495885,2.710294e-10
27,rolling_mean_1h,88292.947047,88292.947047,1.746230e-10


In [12]:
import requests
import pandas as pd
import numpy as np

In [13]:
# ============================================
# Dhanbad Weather for Forecast Period
# ============================================

LATITUDE = 23.796133
LONGITUDE = 86.38477

FORECAST_DATE = "2017-12-31"

weather_url = "https://archive-api.open-meteo.com/v1/archive"

weather_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": FORECAST_DATE,
    "end_date": FORECAST_DATE,
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "cloud_cover",
        "wind_speed_10m"
    ]),
    "timezone": "Asia/Kolkata"
}

response = requests.get(
    weather_url,
    params=weather_params,
    timeout=30
)

response.raise_for_status()

weather_json = response.json()

print("Weather API request successful.")
print("Location:", weather_json.get("latitude"), weather_json.get("longitude"))
print("Timezone:", weather_json.get("timezone"))

Weather API request successful.
Location: 23.796133 86.38477
Timezone: Asia/Kolkata


In [14]:
# ============================================
# Convert API Response to DataFrame
# ============================================

forecast_weather_hourly = pd.DataFrame({
    "Datetime": pd.to_datetime(
        weather_json["hourly"]["time"]
    ),
    "Dhanbad_Temperature": weather_json["hourly"]["temperature_2m"],
    "Dhanbad_Humidity": weather_json["hourly"]["relative_humidity_2m"],
    "Dhanbad_CloudCover": weather_json["hourly"]["cloud_cover"],
    "Dhanbad_WindSpeed": weather_json["hourly"]["wind_speed_10m"]
})

forecast_weather_hourly

,Datetime,Dhanbad_Temperature,Dhanbad_Humidity,Dhanbad_CloudCover,Dhanbad_WindSpeed
0,2017-12-31 00:00:00,14.6,77,68,10.8
1,2017-12-31 01:00:00,13.9,82,64,8.3
2,2017-12-31 02:00:00,13.1,87,59,6.2
3,2017-12-31 03:00:00,12.7,88,62,6.6
4,2017-12-31 04:00:00,12.7,86,65,8.0
5,2017-12-31 05:00:00,12.7,86,33,10.3
6,2017-12-31 06:00:00,13.0,85,0,9.6
7,2017-12-31 07:00:00,14.7,79,0,10.0
8,2017-12-31 08:00:00,17.7,67,1,7.2
9,2017-12-31 09:00:00,20.3,57,2,5.8


In [15]:
print("Shape:", forecast_weather_hourly.shape)

print(
    forecast_weather_hourly["Datetime"].min(),
    "to",
    forecast_weather_hourly["Datetime"].max()
)

print("\nMissing values:")
print(forecast_weather_hourly.isna().sum())

Shape: (24, 5)
2017-12-31 00:00:00 to 2017-12-31 23:00:00

Missing values:
Datetime               0
Dhanbad_Temperature    0
Dhanbad_Humidity       0
Dhanbad_CloudCover     0
Dhanbad_WindSpeed      0
dtype: int64


In [24]:
# ============================================
# Convert Hourly Weather to 30-Minute Weather
# ============================================

# Create the complete 30-minute forecast index
forecast_index = pd.date_range(
    start="2017-12-31 00:00:00",
    end="2017-12-31 23:30:00",
    freq="30min"
)

# Set datetime as index
weather_hourly_indexed = (
    forecast_weather_hourly
    .set_index("Datetime")
)

# Reindex to include the required 30-minute timestamps
weather_30min = weather_hourly_indexed.reindex(
    forecast_index
)

# Interpolate hourly values to 30-minute intervals
weather_30min = (
    weather_30min
    .interpolate(method="time")
)

# Restore datetime column
weather_30min = (
    weather_30min
    .rename_axis("Datetime")
    .reset_index()
)

forecast_weather_30min = weather_30min

print("Shape:", forecast_weather_30min.shape)

print(
    "Range:",
    forecast_weather_30min["Datetime"].min(),
    "to",
    forecast_weather_30min["Datetime"].max()
)

print("\nMissing values:")
print(
    forecast_weather_30min.isna().sum()
)

Shape: (48, 5)
Range: 2017-12-31 00:00:00 to 2017-12-31 23:30:00

Missing values:
Datetime               0
Dhanbad_Temperature    0
Dhanbad_Humidity       0
Dhanbad_CloudCover     0
Dhanbad_WindSpeed      0
dtype: int64


In [17]:
print("Shape:", forecast_weather_30min.shape)

print(
    "Range:",
    forecast_weather_30min["Datetime"].min(),
    "to",
    forecast_weather_30min["Datetime"].max()
)

print("\nMissing values:")
print(forecast_weather_30min.isna().sum())

Shape: (47, 5)
Range: 2017-12-31 00:00:00 to 2017-12-31 23:00:00

Missing values:
Datetime               0
Dhanbad_Temperature    0
Dhanbad_Humidity       0
Dhanbad_CloudCover     0
Dhanbad_WindSpeed      0
dtype: int64


In [25]:
# ============================================
# Weather Validation
# ============================================

weather_columns = [
    "Dhanbad_Temperature",
    "Dhanbad_Humidity",
    "Dhanbad_CloudCover",
    "Dhanbad_WindSpeed"
]

print("Rows:", len(forecast_weather_30min))
print("Expected rows:", 48)

print("\nDuplicate timestamps:")
print(
    forecast_weather_30min["Datetime"].duplicated().sum()
)

print("\nMissing values:")
print(
    forecast_weather_30min[weather_columns].isna().sum()
)

print("\nWeather summary:")
print(
    forecast_weather_30min[weather_columns].describe()
)

Rows: 48
Expected rows: 48

Duplicate timestamps:
0

Missing values:
Dhanbad_Temperature    0
Dhanbad_Humidity       0
Dhanbad_CloudCover     0
Dhanbad_WindSpeed      0
dtype: int64

Weather summary:
       Dhanbad_Temperature  Dhanbad_Humidity  Dhanbad_CloudCover  \
count            48.000000         48.000000            48.00000   
mean             17.917708         66.854167            33.59375   
std               4.397007         17.725435            23.98130   
min              12.700000         38.000000             0.00000   
25%              14.162500         48.500000             9.50000   
50%              16.650000         72.750000            32.75000   
75%              22.150000         82.250000            58.50000   
max              25.200000         88.000000            68.00000   

       Dhanbad_WindSpeed  
count          48.000000  
mean            8.745833  
std             2.605637  
min             4.500000  
25%             6.575000  
50%             8.375000 

In [28]:
FORECAST_WEATHER_PATH = (
    "../data/production/dhanbad_forecast_weather_2017-12-31.csv"
)

forecast_weather_30min.to_csv(
    FORECAST_WEATHER_PATH,
    index=False
)

print("Saved:", FORECAST_WEATHER_PATH)

Saved: ../data/production/dhanbad_forecast_weather_2017-12-31.csv


In [21]:
# ============================================
# Reload Test
# ============================================

test_weather = pd.read_csv(
    FORECAST_WEATHER_PATH,
    parse_dates=["Datetime"]
)

print(test_weather.shape)
print(test_weather.head())
print(test_weather.tail())

(47, 5)
             Datetime  Dhanbad_Temperature  Dhanbad_Humidity  \
0 2017-12-31 00:00:00                14.60              77.0   
1 2017-12-31 00:30:00                14.25              79.5   
2 2017-12-31 01:00:00                13.90              82.0   
3 2017-12-31 01:30:00                13.50              84.5   
4 2017-12-31 02:00:00                13.10              87.0   

   Dhanbad_CloudCover  Dhanbad_WindSpeed  
0                68.0              10.80  
1                66.0               9.55  
2                64.0               8.30  
3                61.5               7.25  
4                59.0               6.20  
              Datetime  Dhanbad_Temperature  Dhanbad_Humidity  \
42 2017-12-31 21:00:00                15.80              80.0   
43 2017-12-31 21:30:00                15.55              80.5   
44 2017-12-31 22:00:00                15.30              81.0   
45 2017-12-31 22:30:00                15.00              82.0   
46 2017-12-31 23:00:00  

In [26]:
forecast_weather_30min.shape

(48, 5)

In [27]:
forecast_weather_30min.head()
forecast_weather_30min.tail()

,Datetime,Dhanbad_Temperature,Dhanbad_Humidity,Dhanbad_CloudCover,Dhanbad_WindSpeed
43,2017-12-31 21:30:00,15.55,80.5,55.0,12.60
44,2017-12-31 22:00:00,15.30,81.0,50.0,11.90
45,2017-12-31 22:30:00,15.00,82.0,39.5,11.55
46,2017-12-31 23:00:00,14.70,83.0,29.0,11.20
47,2017-12-31 23:30:00,14.70,83.0,29.0,11.20


In [29]:
# ============================================
# Forecast Weather Boundary Handling
# ============================================

# Open-Meteo provides hourly observations through 23:00.
# The required forecasting resolution is 30 minutes.
# Therefore, the final 23:30 observation is carried forward
# from the latest available hourly observation (23:00).
#
# This is only a boundary-value treatment and does not
# modify any historical training observations.

forecast_weather_30min.loc[
    forecast_weather_30min["Datetime"] == "2017-12-31 23:30:00",
    [
        "Dhanbad_Temperature",
        "Dhanbad_Humidity",
        "Dhanbad_CloudCover",
        "Dhanbad_WindSpeed"
    ]
] = forecast_weather_30min.loc[
    forecast_weather_30min["Datetime"] == "2017-12-31 23:00:00",
    [
        "Dhanbad_Temperature",
        "Dhanbad_Humidity",
        "Dhanbad_CloudCover",
        "Dhanbad_WindSpeed"
    ]
].values

print("Forecast weather records:", len(forecast_weather_30min))
print("Final timestamp:", forecast_weather_30min["Datetime"].max())
print("Missing values:", forecast_weather_30min.isna().sum().sum())

Forecast weather records: 48
Final timestamp: 2017-12-31 23:30:00
Missing values: 0


In [30]:
forecast_weather_30min.to_csv(
    "../data/production/dhanbad_forecast_weather_2017-12-31.csv",
    index=False
)

print("Forecast weather saved successfully.")

Forecast weather saved successfully.


In [36]:
# ============================================================
# COMPLETE RECURSIVE FORECAST FEATURE TEST
# Run this entire cell at once
# ============================================================

import os
import sys
import importlib
import joblib
import numpy as np
import pandas as pd

# ============================================================
# 1. Configure project path
# ============================================================

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)


# ============================================================
# 2. Reload backend feature-engineering module
# ============================================================

import backend.feature_engineering as feature_engineering

feature_engineering = importlib.reload(
    feature_engineering
)

create_recursive_forecast_features = (
    feature_engineering.create_recursive_forecast_features
)

print("Recursive feature function loaded successfully.")


# ============================================================
# 3. Load integrated historical dataset
# ============================================================

integrated_df = pd.read_csv(
    "data/processed/integrated_demand_weather_holidays.csv",
    parse_dates=["Datetime", "Date"]
)

integrated_df = (
    integrated_df
    .sort_values("Datetime")
    .reset_index(drop=True)
)

print("\nIntegrated dataset:")
print("Shape:", integrated_df.shape)
print(
    "Range:",
    integrated_df["Datetime"].min(),
    "to",
    integrated_df["Datetime"].max()
)


# ============================================================
# 4. Load model metadata
# ============================================================

metadata = joblib.load(
    "models/feature_metadata.pkl"
)

feature_columns = metadata["feature_columns"]

print("\nModel feature count:", len(feature_columns))


# ============================================================
# 5. Create historical load series
# ============================================================

load_history = (
    integrated_df[
        ["Datetime", "Total_Load"]
    ]
    .set_index("Datetime")["Total_Load"]
    .copy()
)

load_history.index = pd.to_datetime(
    load_history.index
)

load_history = load_history.sort_index()

print(
    "Historical load observations:",
    len(load_history)
)


# ============================================================
# 6. Load forecast weather
# ============================================================

FORECAST_WEATHER_PATH = (
    "../data/production/"
    "dhanbad_forecast_weather_2017-12-31.csv"
)

forecast_weather_30min = pd.read_csv(
    FORECAST_WEATHER_PATH,
    parse_dates=["Datetime"]
)

forecast_weather_30min = (
    forecast_weather_30min
    .sort_values("Datetime")
    .reset_index(drop=True)
)

print("\nForecast weather:")
print("Shape:", forecast_weather_30min.shape)
print(
    "Range:",
    forecast_weather_30min["Datetime"].min(),
    "to",
    forecast_weather_30min["Datetime"].max()
)


# ============================================================
# 7. Select first forecast timestamp
# ============================================================

forecast_timestamp = pd.Timestamp(
    "2017-12-31 00:00:00"
)

weather_match = forecast_weather_30min[
    forecast_weather_30min["Datetime"]
    == forecast_timestamp
]

if weather_match.empty:
    raise ValueError(
        f"No weather data found for {forecast_timestamp}"
    )

weather_row = weather_match.iloc[0]


# ============================================================
# 8. Generate the 34 production features
# ============================================================

test_features = create_recursive_forecast_features(
    prediction_datetime=forecast_timestamp,
    historical_df=integrated_df,
    load_history=load_history,
    weather_row=weather_row,
    feature_columns=feature_columns
)


# ============================================================
# 9. Validate generated features
# ============================================================

print("\n============================================")
print("RECURSIVE FEATURE TEST")
print("============================================")

print(
    "Feature shape:",
    test_features.shape
)

print(
    "Number of features:",
    test_features.shape[1]
)

print(
    "Missing values:",
    test_features.isna().sum().sum()
)

print(
    "Feature order correct:",
    list(test_features.columns)
    == list(feature_columns)
)


# ============================================================
# 10. Load trained XGBoost model
# ============================================================

from backend.model_service import model

print("\nTrained model loaded successfully.")


# ============================================================
# 11. Generate first future prediction
# ============================================================

prediction_1 = float(
    model.predict(test_features)[0]
)


# ============================================================
# 12. Display result
# ============================================================

print("\n============================================")
print("FIRST RECURSIVE FORECAST")
print("============================================")

print(
    "Forecast timestamp:",
    forecast_timestamp
)

print(
    f"Predicted load: {prediction_1:,.2f}"
)

print(
    "Temperature:",
    weather_row["Dhanbad_Temperature"]
)

print(
    "Humidity:",
    weather_row["Dhanbad_Humidity"]
)

print(
    "Cloud cover:",
    weather_row["Dhanbad_CloudCover"]
)

print(
    "Wind speed:",
    weather_row["Dhanbad_WindSpeed"]
)

print("\nTEST COMPLETED SUCCESSFULLY.")

Project root: C:\Users\jaina_2j2vl9z\Desktop\exascaleAi\Project
Recursive feature function loaded successfully.

Integrated dataset:
Shape: (17472, 14)
Range: 2017-01-01 00:00:00 to 2017-12-30 23:30:00

Model feature count: 34
Historical load observations: 17472

Forecast weather:
Shape: (48, 5)
Range: 2017-12-31 00:00:00 to 2017-12-31 23:30:00

RECURSIVE FEATURE TEST
Feature shape: (1, 34)
Number of features: 34
Missing values: 0
Feature order correct: True

Trained model loaded successfully.

FIRST RECURSIVE FORECAST
Forecast timestamp: 2017-12-31 00:00:00
Predicted load: 62,170.83
Temperature: 14.6
Humidity: 77.0
Cloud cover: 68.0
Wind speed: 10.8

TEST COMPLETED SUCCESSFULLY.


In [40]:
# ============================================================
# FULL 24-HOUR RECURSIVE FORECAST
# CORRECT PROJECT PATH VERSION
# ============================================================

import os
import sys
import joblib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT ROOT
# ------------------------------------------------------------

# Your Jupyter notebook is running inside:
# C:\Users\jaina_2j2vl9z\Desktop\exascaleAi\Project\Project - 1

PROJECT_ROOT = os.getcwd()

print("Project root:", PROJECT_ROOT)

# Make sure backend can be imported
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ------------------------------------------------------------
# 2. LOAD RECURSIVE FEATURE FUNCTION
# ------------------------------------------------------------

from backend.feature_engineering import (
    create_recursive_forecast_features
)

print("Recursive feature function loaded successfully.")

# ------------------------------------------------------------
# 3. LOAD INTEGRATED DATASET
# ------------------------------------------------------------

INTEGRATED_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "integrated_demand_weather_holidays.csv"
)

print("\nIntegrated data path:")
print(INTEGRATED_PATH)

if not os.path.exists(INTEGRATED_PATH):
    raise FileNotFoundError(
        f"\nIntegrated dataset not found at:\n"
        f"{INTEGRATED_PATH}\n\n"
        f"Current notebook directory:\n"
        f"{PROJECT_ROOT}"
    )

integrated_df = pd.read_csv(
    INTEGRATED_PATH,
    parse_dates=["Datetime", "Date"]
)

integrated_df = (
    integrated_df
    .sort_values("Datetime")
    .reset_index(drop=True)
)

print("\nIntegrated dataset:")
print("Shape:", integrated_df.shape)
print(
    "Range:",
    integrated_df["Datetime"].min(),
    "to",
    integrated_df["Datetime"].max()
)

# ------------------------------------------------------------
# 4. LOAD FEATURE METADATA
# ------------------------------------------------------------

METADATA_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "feature_metadata.pkl"
)

if not os.path.exists(METADATA_PATH):
    raise FileNotFoundError(
        f"Feature metadata not found at:\n{METADATA_PATH}"
    )

feature_metadata = joblib.load(
    METADATA_PATH
)

feature_columns = feature_metadata[
    "feature_columns"
]

print(
    "\nModel feature count:",
    len(feature_columns)
)

# ------------------------------------------------------------
# 5. LOAD TRAINED XGBOOST MODEL
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "final_xgboost_model.pkl"
)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at:\n{MODEL_PATH}"
    )

final_xgb_model = joblib.load(
    MODEL_PATH
)

print(
    "Trained XGBoost model loaded successfully."
)

# ------------------------------------------------------------
# 6. HISTORICAL LOAD
# ------------------------------------------------------------

load_history = (
    integrated_df[
        ["Datetime", "Total_Load"]
    ]
    .copy()
    .set_index("Datetime")
    .sort_index()["Total_Load"]
)

print(
    "Historical load observations:",
    len(load_history)
)

# ------------------------------------------------------------
# 7. LOAD FORECAST WEATHER
# ------------------------------------------------------------

WEATHER_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "production",
    "dhanbad_forecast_weather_2017-12-31.csv"
)

if not os.path.exists(WEATHER_PATH):
    raise FileNotFoundError(
        f"Forecast weather file not found at:\n"
        f"{WEATHER_PATH}"
    )

forecast_weather_30min = pd.read_csv(
    WEATHER_PATH,
    parse_dates=["Datetime"]
)

forecast_weather_30min = (
    forecast_weather_30min
    .sort_values("Datetime")
    .reset_index(drop=True)
)

print("\nForecast weather:")
print("Shape:", forecast_weather_30min.shape)
print(
    "Range:",
    forecast_weather_30min["Datetime"].min(),
    "to",
    forecast_weather_30min["Datetime"].max()
)

# ------------------------------------------------------------
# 8. HOLIDAY LOOKUP
# ------------------------------------------------------------

holiday_lookup = (
    integrated_df[
        [
            "Date",
            "Holiday_Name",
            "Holiday_Type",
            "Scope",
            "Is_Public_Holiday"
        ]
    ]
    .drop_duplicates(subset=["Date"])
    .set_index("Date")
)

# ------------------------------------------------------------
# 9. FORECAST TIMESTAMPS
# ------------------------------------------------------------

forecast_start = pd.Timestamp(
    "2017-12-31 00:00:00"
)

forecast_timestamps = pd.date_range(
    start=forecast_start,
    periods=48,
    freq="30min"
)

# ------------------------------------------------------------
# 10. GENERATE RECURSIVE FORECAST
# ------------------------------------------------------------

forecast_results = []

print("\n" + "=" * 60)
print("GENERATING FULL 24-HOUR RECURSIVE FORECAST")
print("=" * 60)

for i, prediction_datetime in enumerate(
    forecast_timestamps,
    start=1
):

    # --------------------------------------------------------
    # Weather
    # --------------------------------------------------------

    weather_match = forecast_weather_30min[
        forecast_weather_30min["Datetime"]
        == prediction_datetime
    ]

    if weather_match.empty:
        raise ValueError(
            f"No weather data found for "
            f"{prediction_datetime}"
        )

    weather_row = weather_match.iloc[0]

    # --------------------------------------------------------
    # Generate production features
    # --------------------------------------------------------

    features = create_recursive_forecast_features(
        prediction_datetime=prediction_datetime,
        historical_df=integrated_df,
        load_history=load_history,
        weather_row=weather_row,
        feature_columns=feature_columns
    )

    # --------------------------------------------------------
    # Validate features
    # --------------------------------------------------------

    if features.shape != (1, 34):
        raise ValueError(
            f"Unexpected feature shape at "
            f"{prediction_datetime}: "
            f"{features.shape}"
        )

    if features.isna().sum().sum() != 0:
        raise ValueError(
            f"Missing feature values at "
            f"{prediction_datetime}"
        )

    if list(features.columns) != list(feature_columns):
        raise ValueError(
            f"Feature order mismatch at "
            f"{prediction_datetime}"
        )

    # --------------------------------------------------------
    # MODEL PREDICTION
    # --------------------------------------------------------

    predicted_load = float(
        final_xgb_model.predict(features)[0]
    )

    # --------------------------------------------------------
    # RECURSIVE UPDATE
    # --------------------------------------------------------

    load_history.loc[
        prediction_datetime
    ] = predicted_load

    load_history = load_history.sort_index()

    # --------------------------------------------------------
    # HOLIDAY INFORMATION
    # --------------------------------------------------------

    prediction_date = (
        prediction_datetime.normalize()
    )

    if prediction_date in holiday_lookup.index:

        holiday_info = holiday_lookup.loc[
            prediction_date
        ]

        holiday_name = holiday_info[
            "Holiday_Name"
        ]

        holiday_type = holiday_info[
            "Holiday_Type"
        ]

        scope = holiday_info[
            "Scope"
        ]

        is_public_holiday = int(
            holiday_info[
                "Is_Public_Holiday"
            ]
        )

    else:

        holiday_name = None
        holiday_type = None
        scope = None
        is_public_holiday = 0

    # --------------------------------------------------------
    # STORE RESULT
    # --------------------------------------------------------

    forecast_results.append({
        "Datetime": prediction_datetime,

        "Predicted_Load": predicted_load,

        "Dhanbad_Temperature": float(
            weather_row[
                "Dhanbad_Temperature"
            ]
        ),

        "Dhanbad_Humidity": float(
            weather_row[
                "Dhanbad_Humidity"
            ]
        ),

        "Dhanbad_CloudCover": float(
            weather_row[
                "Dhanbad_CloudCover"
            ]
        ),

        "Dhanbad_WindSpeed": float(
            weather_row[
                "Dhanbad_WindSpeed"
            ]
        ),

        "Is_Public_Holiday": is_public_holiday,

        "Holiday_Name": holiday_name,

        "Holiday_Type": holiday_type,

        "Scope": scope
    })

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    print(
        f"{i:02d}/48 | "
        f"{prediction_datetime.strftime('%Y-%m-%d %H:%M')} | "
        f"Predicted Load: "
        f"{predicted_load:,.2f}"
    )

# ------------------------------------------------------------
# 11. CREATE FORECAST DATAFRAME
# ------------------------------------------------------------

forecast_24h_df = pd.DataFrame(
    forecast_results
)

# ------------------------------------------------------------
# 12. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FORECAST VALIDATION")
print("=" * 60)

print(
    "Rows:",
    len(forecast_24h_df)
)

print(
    "Expected rows: 48"
)

print(
    "Range:",
    forecast_24h_df["Datetime"].min(),
    "to",
    forecast_24h_df["Datetime"].max()
)

print(
    "Missing values:",
    forecast_24h_df.isna().sum().sum()
)

print(
    "Duplicate timestamps:",
    forecast_24h_df[
        "Datetime"
    ].duplicated().sum()
)

print(
    "Negative predictions:",
    (
        forecast_24h_df[
            "Predicted_Load"
        ] < 0
    ).sum()
)

print(
    "Minimum prediction:",
    f"{forecast_24h_df['Predicted_Load'].min():,.2f}"
)

print(
    "Maximum prediction:",
    f"{forecast_24h_df['Predicted_Load'].max():,.2f}"
)

print(
    "Mean prediction:",
    f"{forecast_24h_df['Predicted_Load'].mean():,.2f}"
)

# ------------------------------------------------------------
# 13. DISPLAY
# ------------------------------------------------------------

display(
    forecast_24h_df
)

# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

FORECAST_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "production",
    "forecast_2017-12-31.csv"
)

os.makedirs(
    os.path.dirname(FORECAST_PATH),
    exist_ok=True
)

forecast_24h_df.to_csv(
    FORECAST_PATH,
    index=False
)

print("\n" + "=" * 60)
print("FORECAST SAVED SUCCESSFULLY")
print("=" * 60)

print(
    "Saved to:",
    FORECAST_PATH
)

Project root: C:\Users\jaina_2j2vl9z\Desktop\exascaleAi\Project\Project - 1
Recursive feature function loaded successfully.

Integrated data path:
C:\Users\jaina_2j2vl9z\Desktop\exascaleAi\Project\Project - 1\data\processed\integrated_demand_weather_holidays.csv

Integrated dataset:
Shape: (17472, 14)
Range: 2017-01-01 00:00:00 to 2017-12-30 23:30:00

Model feature count: 34
Trained XGBoost model loaded successfully.
Historical load observations: 17472

Forecast weather:
Shape: (48, 5)
Range: 2017-12-31 00:00:00 to 2017-12-31 23:30:00

GENERATING FULL 24-HOUR RECURSIVE FORECAST
01/48 | 2017-12-31 00:00 | Predicted Load: 62,170.83
02/48 | 2017-12-31 00:30 | Predicted Load: 58,045.09
03/48 | 2017-12-31 01:00 | Predicted Load: 54,704.00
04/48 | 2017-12-31 01:30 | Predicted Load: 51,678.26
05/48 | 2017-12-31 02:00 | Predicted Load: 48,955.52
06/48 | 2017-12-31 02:30 | Predicted Load: 46,928.58
07/48 | 2017-12-31 03:00 | Predicted Load: 45,008.65
08/48 | 2017-12-31 03:30 | Predicted Load: 4

,Datetime,Predicted_Load,Dhanbad_Temperature,Dhanbad_Humidity,Dhanbad_CloudCover,Dhanbad_WindSpeed,Is_Public_Holiday,Holiday_Name,Holiday_Type,Scope
0,2017-12-31 00:00:00,62170.828125,14.60,77.0,68.0,10.80,0,None,None,None
1,2017-12-31 00:30:00,58045.093750,14.25,79.5,66.0,9.55,0,None,None,None
2,2017-12-31 01:00:00,54703.996094,13.90,82.0,64.0,8.30,0,None,None,None
3,2017-12-31 01:30:00,51678.257812,13.50,84.5,61.5,7.25,0,None,None,None
4,2017-12-31 02:00:00,48955.523438,13.10,87.0,59.0,6.20,0,None,None,None
5,2017-12-31 02:30:00,46928.578125,12.90,87.5,60.5,6.40,0,None,None,None
6,2017-12-31 03:00:00,45008.652344,12.70,88.0,62.0,6.60,0,None,None,None
7,2017-12-31 03:30:00,43502.121094,12.70,87.0,63.5,7.30,0,None,None,None
8,2017-12-31 04:00:00,42106.976562,12.70,86.0,65.0,8.00,0,None,None,None
9,2017-12-31 04:30:00,40706.902344,12.70,86.0,49.0,9.15,0,None,None,None



FORECAST SAVED SUCCESSFULLY
Saved to: C:\Users\jaina_2j2vl9z\Desktop\exascaleAi\Project\Project - 1\data\production\forecast_2017-12-31.csv


In [41]:
# ============================================================
# FORECAST QUALITY CONTROL
# ============================================================

print("=" * 60)
print("FINAL FORECAST QUALITY CONTROL")
print("=" * 60)

# Required numeric fields
required_columns = [
    "Datetime",
    "Predicted_Load",
    "Dhanbad_Temperature",
    "Dhanbad_Humidity",
    "Dhanbad_CloudCover",
    "Dhanbad_WindSpeed",
    "Is_Public_Holiday"
]

required_missing = (
    forecast_24h_df[required_columns]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing values in required fields:",
    required_missing
)

# Optional holiday description fields
optional_holiday_columns = [
    "Holiday_Name",
    "Holiday_Type",
    "Scope"
]

optional_missing = (
    forecast_24h_df[optional_holiday_columns]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing values in optional holiday fields:",
    optional_missing
)

# Timestamp validation
expected_timestamps = pd.date_range(
    "2017-12-31 00:00:00",
    periods=48,
    freq="30min"
)

timestamps_correct = (
    forecast_24h_df["Datetime"].tolist()
    == expected_timestamps.tolist()
)

print(
    "48 timestamps correct:",
    timestamps_correct
)

# Prediction validation
predictions_valid = (
    forecast_24h_df["Predicted_Load"].notna().all()
    and
    (forecast_24h_df["Predicted_Load"] >= 0).all()
)

print(
    "Predictions valid:",
    predictions_valid
)

# Weather validation
weather_columns = [
    "Dhanbad_Temperature",
    "Dhanbad_Humidity",
    "Dhanbad_CloudCover",
    "Dhanbad_WindSpeed"
]

weather_missing = (
    forecast_24h_df[weather_columns]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing weather values:",
    weather_missing
)

# Duplicate check
duplicates = forecast_24h_df[
    "Datetime"
].duplicated().sum()

print(
    "Duplicate timestamps:",
    duplicates
)

# Final status
if (
    required_missing == 0
    and timestamps_correct
    and predictions_valid
    and weather_missing == 0
    and duplicates == 0
):
    print("\n" + "=" * 60)
    print("FINAL QC: PASSED")
    print("=" * 60)
else:
    print("\n" + "=" * 60)
    print("FINAL QC: FAILED")
    print("=" * 60)

FINAL FORECAST QUALITY CONTROL
Missing values in required fields: 0
Missing values in optional holiday fields: 144
48 timestamps correct: True
Predictions valid: True
Missing weather values: 0
Duplicate timestamps: 0

FINAL QC: PASSED


In [42]:
# ============================================================
# TEST FORECAST SERVICE
# ============================================================

import sys
import os

PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from backend.forecast_service import (
    generate_24h_forecast
)

print("Forecast service imported successfully.")

result = generate_24h_forecast(
    "2017-12-31 00:00:00"
)

print("\n" + "=" * 60)
print("FORECAST SERVICE TEST")
print("=" * 60)

print(
    "Start:",
    result["forecast_start"]
)

print(
    "End:",
    result["forecast_end"]
)

print(
    "Interval:",
    result["interval_minutes"],
    "minutes"
)

print(
    "Forecast points:",
    result["forecast_points"]
)

print(
    "\nFirst prediction:"
)

print(
    result["forecast"][0]
)

print(
    "\nLast prediction:"
)

print(
    result["forecast"][-1]
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert result["forecast_points"] == 48

assert (
    result["interval_minutes"]
    == 30
)

assert len(
    result["forecast"]
) == 48

assert all(
    point["predicted_load"] >= 0
    for point in result["forecast"]
)

print("\n" + "=" * 60)
print("FORECAST SERVICE TEST PASSED")
print("=" * 60)

Forecast service imported successfully.

FORECAST SERVICE TEST
Start: 2017-12-31 00:00:00
End: 2017-12-31 23:30:00
Interval: 30 minutes
Forecast points: 48

First prediction:
{'datetime': '2017-12-31 00:00:00', 'predicted_load': 62170.828125, 'temperature': 14.6, 'humidity': 77.0, 'cloud_cover': 68.0, 'wind_speed': 10.8, 'is_public_holiday': 0, 'holiday_name': None, 'holiday_type': None, 'scope': None}

Last prediction:
{'datetime': '2017-12-31 23:30:00', 'predicted_load': 67133.4375, 'temperature': 14.7, 'humidity': 83.0, 'cloud_cover': 29.0, 'wind_speed': 11.2, 'is_public_holiday': 0, 'holiday_name': None, 'holiday_type': None, 'scope': None}

FORECAST SERVICE TEST PASSED
